# Synthetic optimizer SQL companion
## Goal
Independent AI-assisted proof exercise using three invented optimizer runs and
four checkpoints. SQL source and retained outputs make the design inspectable.
This is independent proof work, not employment or production experience.
## Setup
Use this folder as the working directory. Python uses its standard library.
Database execution requires an existing authorized local PostgreSQL database and
psql on PATH; this public package includes no binaries, credentials or cluster.
## Execution record
Five code cells were executed top-to-bottom in one Python 3.10.11 namespace on
2026-10-01. This was a standard-library runner, not a Jupyter kernel or nbclient.
Retained database outputs report PostgreSQL 18.6. Host paths have been redacted;
computational source and numeric results are unchanged. This public review did
not rerun the PostgreSQL pipeline. Timing values below belong to the notebook's
pass; postgres_results.txt contains the later repeated pass.


In [1]:
from pathlib import Path
import companion
assert Path('schema.sql').exists(), 'Start notebook in sql-proof directory'


## Steps
### Prepare synthetic inputs
Input grain: run and numbered checkpoint. Loss is minimized; allocation is
per run. See README for batch identity, limits and transaction boundaries.


In [2]:
# Deterministic data; writes only fixture, seed SQL and expected CSV.
companion.prepare()
payload = companion.fixture()
print(len(payload['runs']), 'runs;', len(payload['checkpoints']), 'checkpoints')


3 runs; 4 checkpoints


### Independent hand calculation
For r1, (100 - 60) / 10 = 4. For r2, (100 - 60) / 20 = 2.
r3 has no checkpoints and zero budget: best loss 50, gain 0, ratio NULL.
The table is a Python oracle, not observed database output.


In [3]:
for row in companion.oracle(payload):
    print(row)


{'run_id': 'r1', 'method': 'synthetic-a', 'budget': 10, 'best_loss': 60, 'improvement': Decimal('40'), 'improvement_per_budget': Decimal('4')}
{'run_id': 'r2', 'method': 'synthetic-b', 'budget': 20, 'best_loss': 60, 'improvement': Decimal('40'), 'improvement_per_budget': Decimal('2')}
{'run_id': 'r3', 'method': 'synthetic-zero', 'budget': 0, 'best_loss': 50, 'improvement': Decimal('0'), 'improvement_per_budget': None}


## Checks
### Validate local fixture and oracle
These checks do not validate PostgreSQL semantics or execute SQL.


In [4]:
companion.check()


PASS: fixture counts; Decimal oracle; empty input; deterioration; CSV; notebook code syntax.
These are Python checks, not PostgreSQL execution.


### Execute the real PostgreSQL pipeline when available
The next cell executes in the separately saved companion.executed.ipynb.
It fails explicitly if psql is absent. It writes real version/test/plan evidence
and actual_summary.csv only after connecting to the authorized local database.


In [5]:
# Requires existing local authorization and libpq connection settings.
companion.run()
print(Path('actual_summary.csv').read_text())
# Keep notebook evidence bounded: assertion notices and measured timing only.
for line in Path('postgres_results.txt').read_text().splitlines():
    if 'PASS' in line or 'Planning Time:' in line or 'Execution Time:' in line:
        print(line)


PASS PostgreSQL assertions and independently reconciled actual_summary.csv.
run_id,method,budget,best_loss,improvement,improvement_per_budget
r1,synthetic-a,10,60,40,4.0000000000000000
r2,synthetic-b,20,60,40,2.0000000000000000
r3,synthetic-zero,0,50,0,

psql:[project]/tests.sql:35: NOTICE:  PASS exact replay idempotency
psql:[project]/tests.sql:35: NOTICE:  PASS function hand calculation and zero budget
psql:[project]/tests.sql:35: NOTICE:  PASS no-checkpoint fallback
psql:[project]/tests.sql:35: NOTICE:  PASS empty and unknown batch
psql:[project]/tests.sql:35: NOTICE:  PASS known CTE/window result
psql:[project]/tests.sql:62: NOTICE:  PASS rejection and rollback: test-orphan (SQLSTATE P0001)
psql:[project]/tests.sql:64: NOTICE:  PASS rejection and rollback: test-duplicate-run (SQLSTATE 23505)
psql:[project]/tests.sql:66: NOTICE:  PASS rejection and rollback: test-late-failure (SQLSTATE 23514)
psql:[project]/tests.sql:68: NOTICE:  PASS rejection and rollback: test-duplicate-checkpoin

## Next steps
Inspect schema.sql, tests.sql, analysis.sql, postgres_results.txt and the CSVs.
Explain exact replay identity, transaction rollback, the run/checkpoint grain,
NULL handling for zero budget, and the running-minimum window frame.
These tiny-fixture observations do not establish throughput or scalability.
